# Caso D · 03 Features para predicción de ocupación

> _Tutorial · Caso de uso: **D — IAQ + ocupación** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir features informativas para detectar ocupación a partir de variables ambientales (sin sensor de presencia).


## 2. Qué se aprende

- Derivada del CO₂ (`dCO2/dt`).
- IAQ index sintético.
- Lag features y agregados ventana.
- Cómo manejar el desbalance de clases.


## 3. Contexto del caso de uso

El sensor de presencia no siempre existe; queremos inferir ocupación indirectamente desde ambiente (CO₂, sonido).


## 4. Relación con CENTINELA+

Si AULA01 no tiene sensor de presencia, este modelo lo sustituye.


## 5. Relación con Medallion

Lee plata, escribe oro local.


## 6. Datos de entrada

Mock In-Gauge.


## 7. Schema CAPTIA esperado

No aplica (oro).


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Reusamos el CSV.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "ingauge_aula01_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).set_index("timestamp")
df.head()


## 10. Exploración paso a paso

Computamos features.


In [ ]:
def make_features(d):
    f = pd.DataFrame(index=d.index)
    f["co2"] = d["Indoor_CO2"]
    f["temp"] = d["Indoor_Temp"]
    f["rh"] = d["Indoor_Hum"]
    f["lux"] = d["Indoor_Lux"]
    f["noise"] = d["Indoor_Noise"]
    f["dco2_5min"] = d["Indoor_CO2"].diff(5)
    f["co2_lag_15"] = d["Indoor_CO2"].shift(15)
    f["co2_roll_30"] = d["Indoor_CO2"].rolling(30).mean()
    f["noise_roll_15"] = d["Indoor_Noise"].rolling(15).mean()
    # IAQ aproximado
    f["iaq"] = (
        50 * (f["co2"] / 1000).clip(upper=5)
        + 30 * (f["rh"] / 50 - 1).abs()
        + 15 * (f["temp"] / 22 - 1).abs() * 100
    ).clip(upper=500)
    f["y_occupied"] = d["Occupied"]
    return f.dropna()

X = make_features(df)
X.head()


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Persistimos.


In [ ]:
out_dir = ROOT / "output" / "case_D"
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / "iaq_features.parquet"
X.to_parquet(parquet_path)
print(f"Wrote {parquet_path.relative_to(ROOT)} ({len(X)})")


## 13. Visualizaciones explicativas

Comparativa correlaciones.


In [ ]:
correls = X.drop(columns=["y_occupied"]).corrwith(X["y_occupied"]).sort_values()
correls.plot.barh(color="#FF5722", figsize=(7, 4))
plt.title("Correlación con y_occupied")
plt.tight_layout()


## 14. Validaciones

El `y_occupied` mantiene la fracción esperada (~30%).


In [ ]:
ratio = X["y_occupied"].mean()
print({"y_ratio": ratio})
assert 0.05 < ratio < 0.7


## 15. Errores comunes

1. **Confundir Indoor_Occupancy (0/1) con People_Count (0..N)**.
2. **Suavizar features que cambian rápido** (perdemos picos CO2).
3. **Mezclar lectivo y vacaciones**: meter una columna `is_holiday`.


## 16. Ejercicios propuestos

1. Añade `holiday` (calendario Comunidad Valenciana) y mide ganancia.
2. Prueba `dco2_15min` vs `dco2_5min`.
3. Calcula y plotea IAQ index a lo largo de un día lectivo.


## 17. Cómo se reutiliza con datos reales

Misma `make_features(df)` sobre datos reales — solo cambia el path.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `04_case_D_iaq_occupancy/04_modelo_ocupacion_desde_ambiente.ipynb`.
- Documento web del caso: `docs/use-cases/case-d-iaq-occupancy.md`.
